In [ ]:
!pip install -U zarr s3fs fsspec xarray

In [1]:
import xarray as xr

print(xr.backends.list_engines())

{'scipy': <ScipyBackendEntrypoint>
  Open netCDF files (.nc, .cdf and .nc.gz) using scipy in Xarray
  Learn more at https://docs.xarray.dev/en/stable/generated/xarray.backends.ScipyBackendEntrypoint.html, 'store': <StoreBackendEntrypoint>
  Open AbstractDataStore instances in Xarray
  Learn more at https://docs.xarray.dev/en/stable/generated/xarray.backends.StoreBackendEntrypoint.html, 'zarr': <ZarrBackendEntrypoint>
  Open zarr files (.zarr) using zarr in Xarray
  Learn more at https://docs.xarray.dev/en/stable/generated/xarray.backends.ZarrBackendEntrypoint.html}


In [2]:
import xarray as xr
import fsspec

url = (
    "s3://ncar-na-cordex/day/"
    "prec.rcp45.day.NAM-44i.raw.zarr"
)

mapper = fsspec.get_mapper(
    url,
    anon=True
)

rcp45_prec = xr.open_zarr(
    mapper,
    consolidated=True
)

print(rcp45_prec)

<xarray.Dataset> Size: 32GB
Dimensions:    (member_id: 6, time: 34698, lat: 129, lon: 300, bnds: 2)
Coordinates:
  * member_id  (member_id) <U21 504B 'MPI-ESM-LR.CRCM5-UQAM' ... 'CanESM2.Can...
  * time       (time) datetime64[ns] 278kB 2006-01-01T12:00:00 ... 2100-12-31...
  * lat        (lat) float64 1kB 12.25 12.75 13.25 13.75 ... 75.25 75.75 76.25
  * lon        (lon) float64 2kB -171.8 -171.2 -170.8 ... -23.25 -22.75 -22.25
    time_bnds  (time, bnds) datetime64[ns] 555kB dask.array<chunksize=(17349, 2), meta=np.ndarray>
Dimensions without coordinates: bnds
Data variables:
    prec       (member_id, time, lat, lon) float32 32GB dask.array<chunksize=(4, 1000, 65, 150), meta=np.ndarray>
Attributes: (12/23)
    CORDEX_domain:                  NAM-44
    contact:                        {"MPI-ESM-LR.CRCM5-UQAM": "Winger.Katja@u...
    creation_date:                  {"MPI-ESM-LR.CRCM5-UQAM": "2012-09-19", "...
    driving_experiment:             {"MPI-ESM-LR.CRCM5-UQAM": "MPI-M-MPI-ESM

In [3]:


frequency = 'day'
resolution = '44i'
bias_correction = 'raw'




In [4]:
zarr_urls = {
    'rcp45_prec': (
        f's3://ncar-na-cordex/day/'
        f'prec.rcp45.{frequency}.NAM-{resolution}.{bias_correction}.zarr'
    ),

    'rcp85_prec': (
        f's3://ncar-na-cordex/day/'
        f'prec.rcp85.{frequency}.NAM-{resolution}.{bias_correction}.zarr'
    ),

    'rcp45_temp': (
        f's3://ncar-na-cordex/day/'
        f'temp.rcp45.{frequency}.NAM-{resolution}.{bias_correction}.zarr'
    ),

    'rcp85_temp': (
        f's3://ncar-na-cordex/day/'
        f'temp.rcp85.{frequency}.NAM-{resolution}.{bias_correction}.zarr'
    )
}

datasets = {}

for key, url in zarr_urls.items():

    print("Opening:", key)

    mapper = fsspec.get_mapper(
        url,
        anon=True
    )

    datasets[key] = xr.open_zarr(
        mapper,
        consolidated=True
    )

    print("Opened:", key)

Opening: rcp45_prec
Opened: rcp45_prec
Opening: rcp85_prec
Opened: rcp85_prec
Opening: rcp45_temp
Opened: rcp45_temp
Opening: rcp85_temp
Opened: rcp85_temp


In [5]:
lat = slice(37, 41)
lon = slice(-109.1, -105)

In [6]:
# Simulations to use
simulations = [
    'CanESM2.CanRCM4',
    'CanESM2.RCA4'
]

# Study area: western Colorado
lat_bounds = slice(37, 41)
lon_bounds = slice(-109.1, -105)

In [7]:
rcp45_prec_ds = (
    datasets['rcp45_prec']
    .sel(
        lat=lat_bounds,
        lon=lon_bounds,
        member_id=simulations,
        bnds=0
    )
    .drop_vars('time_bnds', errors='ignore')
    .mean(dim=['lat', 'lon'])
)

rcp85_prec_ds = (
    datasets['rcp85_prec']
    .sel(
        lat=lat_bounds,
        lon=lon_bounds,
        member_id=simulations,
        bnds=0
    )
    .drop_vars('time_bnds', errors='ignore')
    .mean(dim=['lat', 'lon'])
)

rcp45_temp_ds = (
    datasets['rcp45_temp']
    .sel(
        lat=lat_bounds,
        lon=lon_bounds,
        member_id=simulations,
        bnds=0
    )
    .drop_vars('time_bnds', errors='ignore')
    .mean(dim=['lat', 'lon'])
)

rcp85_temp_ds = (
    datasets['rcp85_temp']
    .sel(
        lat=lat_bounds,
        lon=lon_bounds,
        member_id=simulations,
        bnds=0
    )
    .drop_vars('time_bnds', errors='ignore')
    .mean(dim=['lat', 'lon'])
)

In [8]:
from pathlib import Path

out_dir = Path("data/scenario/NA_CORDEX")
out_dir.mkdir(parents=True, exist_ok=True)

ds_dict = {
    'rcp45_prec': rcp45_prec_ds,
    'rcp85_prec': rcp85_prec_ds,
    'rcp45_temp': rcp45_temp_ds,
    'rcp85_temp': rcp85_temp_ds
}

for name, ds in ds_dict.items():

    df = ds.to_dataframe().reset_index()

    out_file = out_dir / f"{name}_daily.parquet"

    df.to_parquet(
        out_file,
        index=False
    )

    print("Saved:", out_file)

Saved: data\scenario\NA_CORDEX\rcp45_prec_daily.parquet
Saved: data\scenario\NA_CORDEX\rcp85_prec_daily.parquet
Saved: data\scenario\NA_CORDEX\rcp45_temp_daily.parquet
Saved: data\scenario\NA_CORDEX\rcp85_temp_daily.parquet
